# Lab L3.3 · Card-fraud detection on 1.5M transactions — velocity features, decision trees and random forests
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 3 · ⏱ 80 min · **Level 1 · From scratch**

Engineer real-time-safe velocity features at scale, grow and prune interpretable trees, beat them with a random forest, and evaluate against the fraud team’s daily alert budget.

Follow **L3.3_Lab_Guide.html** step by step. This notebook gives you only the structure — every line of logic is yours. Keep variable names as suggested in the guide so later steps (and later labs) line up.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load 1.5M transactions efficiently and split in time

1. Load with `txn_ts` parsed; cast `merchant_category`, `channel`, `merchant_country`, `customer_home_emirate` to `category`.
2. Print rows, memory (MB), fraud rate, and fraud rate by channel.
3. Define the split dates: train < 2025-08-01 ≤ valid < 2025-09-01 ≤ test.

In [ ]:
# Your code for step 1


## Step 2 · Velocity features — the fraud analyst’s secret weapon

1. Write `build_fraud_features(tx)`: sort by card and time; add `secs_since_prev`, `txn_count_1h`, `txn_count_24h`, `amount_sum_24h` (time-based rolling per card), `amount_vs_card_mean` (vs the card’s *prior* mean), first-time-country and first-time-merchant-category flags, `card_txn_index`, `hour`, `is_night`, `is_international`, `amount_to_salary`, `log_amount`.
2. Apply it; time it. Add `split`.
3. Compare the median of 3 velocity features for fraud vs legitimate transactions.

In [ ]:
# Your code for step 2


## Step 3 · An interpretable decision tree — and its rules

1. Build `prep` = ColumnTransformer(numeric: `SimpleImputer(fill_value=-1)`; categorical: `OrdinalEncoder` unknown→−1).
2. Fit a depth-3 `DecisionTreeClassifier` (Gini) on train; plot it and print `export_text` rules.
3. Write `pr_auc` and `at_budget(y, p, amount, budget=0.005)` (precision, recall and AED share caught in the top 0.5%). Evaluate on valid; repeat with `criterion='entropy'`.

In [ ]:
# Your code for step 3


## Step 4 · Depth, leaf size and cost-complexity pruning

1. For depths {2, 4, 6, 8, 12, 16, None} record train and valid PR-AUC; plot both.
2. On a 150k stratified train sample compute `cost_complexity_pruning_path`; evaluate 8 alphas on valid PR-AUC and keep the best pruned tree.
3. Add the best depth-limited and best pruned trees to `board`.

In [ ]:
# Your code for step 4


## Step 5 · Random forest on an undersampled training set

1. Build `tr_u`: all training fraud + a 250k random sample of training legit.
2. Fit `RandomForestClassifier(n_estimators=300, min_samples_leaf=10, max_features='sqrt', n_jobs=-1)` inside a Pipeline with `prep`; time it.
3. Evaluate at budget on valid; add a `LogisticRegression` baseline on the same features for comparison.
4. Correct probabilities for the sampling ratio β: `p = β·p_s / (β·p_s − p_s + 1)` and compare mean predicted vs actual fraud rate.

In [ ]:
# Your code for step 5


## Step 6 · Which features drive the forest? Impurity vs permutation importance

1. Plot the RF’s impurity importances.
2. Compute `permutation_importance` on a validation sample (all fraud + 40k legit), scoring `average_precision`, `n_repeats=3`.
3. Compare the two top-10 lists side by side.

In [ ]:
# Your code for step 6


## Step 7 · Test-month evaluation and hand-off

1. Score `te` with the forest; report `at_budget` on test and the number of alerts per day.
2. Save `tf` (ids, timestamps, FEATS, target, split, amount) to `CKPT/card_features.parquet`.
3. Save the RF pipeline and `fraud_metrics.json`.

In [ ]:
# Your code for step 7
